# Ensemble Learning

## 1. Gradient Boosting Tree

### 1.1. GBT Regressor

In [1]:
import numpy as np
import pandas as pd
!pip install ucimlrepo
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, f1_score
from xgboost import XGBRegressor, XGBClassifier

# Загружаем датасет для регрессии (качество вина) https://archive.ics.uci.edu/dataset/186/wine+quality
dataset = fetch_ucirepo(id=186)
X = dataset.data.features
y = dataset.data.targets
y = y.values.ravel()   # превращаем в одномерный массив

In [ ]:
# Посмотрим на данные
# print("Признаки:", X.shape)
# print("Целевая переменная:", y.shape, "| от", y.min(), "до", y.max())
# print("Пропусков в X:", X.isnull().sum().sum())
# X.head()

In [2]:
# Split into train and test sets (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X.to_numpy(), y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape, "| Test:", X_test.shape)

Train: (5197, 11) | Test: (1300, 11)


In [3]:
# Create the GBT Regressor class
class GBTRegressor:
    def __init__(self, n_estimators=200, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators    # сколько деревьев строим
        self.learning_rate = learning_rate  # насколько сильно доверяем каждому дереву
        self.max_depth = max_depth          # глубина каждого дерева

    def fit(self, X, y):
        # Шаг 1: начальное предсказание - среднее значение y
        self.init_ = y.mean()
        pred = np.full(len(y), self.init_)
        self.trees_ = []

        for _ in range(self.n_estimators):
            # Шаг 2: считаем остатки (ошибки), то есть сколько ещё не угадали
            residual = y - pred

            # Шаг 3: обучаем маленькое дерево предсказывать эти остатки
            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X, residual)

            # Шаг 4: обновляем предсказание небольшим шагом (learning_rate)
            pred += self.learning_rate * tree.predict(X)
            self.trees_.append(tree)
        return self

    def predict(self, X):
        # Предсказание = среднее + вклад всех деревьев
        pred = np.full(len(X), self.init_)
        for tree in self.trees_:
            pred += self.learning_rate * tree.predict(X)
        return pred

In [5]:
# Словарь, куда будем складывать результаты, чтобы потом сравнить


reg = GBTRegressor(n_estimators=200, learning_rate=0.1, max_depth=3).fit(X_train, y_train)
pred = reg.predict(X_test)
print(f"Custom GBT   RMSE: {np.sqrt(mean_squared_error(y_test, pred)):.4f}  R2: {r2_score(y_test, pred):.4f}")

Custom GBT   RMSE: 0.6706  R2: 0.3910


In [6]:
# Compare with sklearn's GradientBoostingRegressor
sk_reg = GradientBoostingRegressor(
    n_estimators=200, learning_rate=0.1, max_depth=3, random_state=42
).fit(X_train, y_train)
pred = sk_reg.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)
print(f"sklearn GBR  RMSE: {rmse:.4f}  R2: {r2:.4f}")

sklearn GBR  RMSE: 0.6702  R2: 0.3918


In [7]:
# Как качество зависит от числа деревьев (на нашей реализации)
for n in [10, 50, 100, 200]:
    model = GBTRegressor(n_estimators=n, learning_rate=0.1, max_depth=3).fit(X_train, y_train)
    p = model.predict(X_test)
    print("Деревьев:", n, "| RMSE:", round(np.sqrt(mean_squared_error(y_test, p)), 4))

Деревьев: 10 | RMSE: 0.7481
Деревьев: 50 | RMSE: 0.693
Деревьев: 100 | RMSE: 0.6797
Деревьев: 200 | RMSE: 0.6707


### 1.2. GBT Classifier

In [8]:
# Load classification dataset https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic
dataset = fetch_ucirepo(id=17)
X = dataset.data.features
y = dataset.data.targets
y = y.values.ravel()          # превращаем в одномерный массив
y = (y == 'M').astype(int)    # M (злокачественная) = 1, B (доброкачественная) = 0

print("Признаки:", X.shape)
print("Классы (0 - B, 1 - M):", np.bincount(y))

Признаки: (569, 30)
Классы (0 - B, 1 - M): [357 212]


In [9]:
# Split into train and test sets
# stratify=y сохраняет одинаковую долю классов в train и test
X_train, X_test, y_train, y_test = train_test_split(
    X.to_numpy(), y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape, "| Test:", X_test.shape)

Train: (455, 30) | Test: (114, 30)


In [10]:
# Create the GBT Classifier class
class GBTClassifier:
    def __init__(self, n_estimators=200, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth

    def fit(self, X, y):
        # Та же идея, что и у регрессора: предсказываем число (близкое к 0 или 1)
        # и каждым деревом исправляем ошибку.
        self.init_ = y.mean()
        pred = np.full(len(y), self.init_)
        self.trees_ = []

        for _ in range(self.n_estimators):
            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X, y - pred)
            pred += self.learning_rate * tree.predict(X)
            self.trees_.append(tree)
        return self

    def predict(self, X):
        pred = np.full(len(X), self.init_)
        for tree in self.trees_:
            pred += self.learning_rate * tree.predict(X)
        # если число >= 0.5, то класс 1, иначе класс 0
        return (pred >= 0.5).astype(int)

In [11]:
clf_results = {}

# Train the GBT Classifier and evaluate its performance
clf = GBTClassifier(n_estimators=200, learning_rate=0.1, max_depth=3).fit(X_train, y_train)
pred = clf.predict(X_test)

acc = accuracy_score(y_test, pred)
f1 = f1_score(y_test, pred)
clf_results['Наш GBT'] = [acc, f1]
print(f"Custom GBT   Accuracy: {acc:.4f}  F1: {f1:.4f}")

Custom GBT   Accuracy: 0.9561  F1: 0.9367


In [12]:
# Compare with sklearn's GradientBoostingClassifier
sk_clf = GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.1, max_depth=3, random_state=42
).fit(X_train, y_train)
pred = sk_clf.predict(X_test)

acc = accuracy_score(y_test, pred)
f1 = f1_score(y_test, pred)
clf_results['sklearn GBC'] = [acc, f1]
print(f"sklearn GBC  Accuracy: {acc:.4f}  F1: {f1:.4f}")

sklearn GBC  Accuracy: 0.9649  F1: 0.9500


## 2. XGBoost
XGBoost это быстрая и улучшенная реализация градиентного бустинга (есть регуляризация, параллельные вычисления). Используем те же параметры и те же разбиения train/test, что и раньше, чтобы сравнение было честным.

In [13]:
# XGBoost Regressor (качество вина)
data = fetch_ucirepo(id=186)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    data.data.features.to_numpy(), data.data.targets.values.ravel(), test_size=0.2, random_state=42
)

xgb_reg = XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=3, random_state=42)
xgb_reg.fit(Xr_train, yr_train)
pred = xgb_reg.predict(Xr_test)

rmse = np.sqrt(mean_squared_error(yr_test, pred))
r2 = r2_score(yr_test, pred)
print(f"XGBRegressor  RMSE: {rmse:.4f}  R2: {r2:.4f}")

XGBRegressor  RMSE: 0.6699  R2: 0.3923


In [14]:
# XGBoost Classifier (рак молочной железы)
data = fetch_ucirepo(id=17)
yc = (data.data.targets.values.ravel() == "M").astype(int)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    data.data.features.to_numpy(), yc, test_size=0.2, random_state=42, stratify=yc
)

xgb_clf = XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=3,
                        random_state=42, eval_metric="logloss")
xgb_clf.fit(Xc_train, yc_train)
pred = xgb_clf.predict(Xc_test)

acc = accuracy_score(yc_test, pred)
f1 = f1_score(yc_test, pred)
clf_results['XGBClassifier'] = [acc, f1]
print(f"XGBClassifier Accuracy: {acc:.4f}  F1: {f1:.4f}")

XGBClassifier Accuracy: 0.9561  F1: 0.9367


## 3. Сравнение всех моделей

In [15]:
# Таблица для регрессии (wine quality): чем меньше RMSE и чем больше R2, тем лучше
reg_table = pd.DataFrame(reg_results, index=['RMSE', 'R2']).T.round(4)
reg_table

,RMSE,R2
Наш GBT,0.6714,0.3896


In [16]:
# Таблица для классификации (breast cancer): чем больше, тем лучше
clf_table = pd.DataFrame(clf_results, index=['Accuracy', 'F1']).T.round(4)
clf_table

,Accuracy,F1
Наш GBT,0.9561,0.9367
sklearn GBC,0.9649,0.9500
XGBClassifier,0.9561,0.9367
